In [2]:
# Add project root (the folder that contains "src/") to sys.path
import sys
import time
from pathlib import Path

def add_project_root(marker_dir="src", max_hops=5):
    p = Path.cwd().resolve()
    for _ in range(max_hops):
        if (p / marker_dir).exists():
            sys.path.insert(0, str(p))
            print(f"[OK] Added to sys.path: {p}")
            return
        p = p.parent
    raise RuntimeError(f"Could not find '{marker_dir}' within {max_hops} parents from {Path.cwd()}")

add_project_root()  

[OK] Added to sys.path: D:\Data\Projects\Thesis\Adaptive-Hierarchical-Feature-Modulation-U-Net-Model-for-Retinal-Vessel-Segmentation


In [4]:
from pathlib import Path
from tqdm import tqdm

from src.retina_biomarkers.notebook_utils.pipeline.config import PipelineConfig
from src.retina_biomarkers.pipeline.manifest_aptos import build_aptos_manifest
from src.retina_biomarkers.pipeline.stage1_preprocess import make_run_id, stage1_preprocess_one

# ---- paths ----
APTOS_ROOT = Path("../../data/raw/APTOS2019")  
CACHE_ROOT = Path("../../outputs/aptos_cache") # will auto-create

# ---- config ----
cfg = PipelineConfig(
    image_size=512,
    threshold=0.5,
    use_fov_in_model=False,
)

run_id = make_run_id(cfg, prefix="aptos2019")
print("RUN ID:", run_id)

items = build_aptos_manifest(APTOS_ROOT)
print("Found items:", len(items))

# ---- run stage1 (resume-able) ----
ok = skip = fail = 0
for it in tqdm(items):
    try:
        r = stage1_preprocess_one(
            image_id=it.image_id,
            color_path=it.color_path,
            fov_path=it.fov_path,
            fovea_path=it.fovea_path,
            cfg=cfg,
            cache_root=CACHE_ROOT,
            run_id=run_id,
            save_debug_png=True,   # set True for first ~20 samples to sanity check
            overwrite=False
        )
        if r["status"] == "ok": ok += 1
        else: skip += 1
    except Exception as e:
        fail += 1
        print("[FAIL]", it.image_id, e)

print(f"done. ok={ok}, skipped={skip}, fail={fail}")
print("cache folder:", (CACHE_ROOT / run_id / "stage1"))


RUN ID: aptos2019_stage1_03b974d569
Found items: 3665


  9%|▉         | 328/3665 [00:13<02:19, 23.90it/s]


KeyboardInterrupt: 